# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a dataset defined via a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset is loaded from the Croissant schema URL below:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets (tables), their fields and IDs. 
All entities are referenced by their `@id`.

In [ ]:
from pprint import pprint

# List all record sets in the dataset
record_sets = dataset.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"- @id: {rs.id} | name: {rs.name}")

# For each record set, list its fields
for rs in record_sets:
    print(f"\nRecord Set '@id': {rs.id}, name: {rs.name}")
    print("Fields:")
    for field in rs.fields:
        print(f"  - @id: {field.id} | name: {field.name} | dataType: {field.data_type}")

## 3. Data Extraction
Load data from each record set into DataFrames. All references are via their `@id`.

In [ ]:
# Gather record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"\nColumns for record set '@id': {record_set_id}")
    print(dataframes[record_set_id].columns.tolist())
    print("Sample records:")
    display(dataframes[record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization and grouping. 
This example processes the clinical features record set.

In [ ]:
# Choose a record set and a numeric field for demonstration

# Example: Table 1 - clinical features
# Suppose its @id is 'http://sen.science/cr/recordSet/table1', and the field '@id' for age is 'age_at_diagnosis'

clinical_rs_id = None
age_field_id = None
for rs in dataset.record_sets:
    if 'age' in rs.name.lower() or 'table 1' in rs.name.lower():
        clinical_rs_id = rs.id
        for field in rs.fields:
            if 'age' in field.name.lower():
                age_field_id = field.id
        break

# If not found, use the first record set and first numeric field
if clinical_rs_id is None:
    clinical_rs_id = record_set_ids[0]
    for field in dataset.record_sets[0].fields:
        if field.data_type.lower() in ['integer','float','number','double']:
            age_field_id = field.id
            break

df = dataframes[clinical_rs_id]

# Filter records by age > threshold
threshold = 25
if age_field_id in df.columns:
    filtered_df = df[df[age_field_id] > threshold]
    print(f"Filtered records with {age_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize age field
    filtered_df[f"{age_field_id}_normalized"] = (filtered_df[age_field_id] - filtered_df[age_field_id].mean()) / filtered_df[age_field_id].std()
    print(f"Normalized {age_field_id} for filtered records:")
    print(filtered_df[[age_field_id, f"{age_field_id}_normalized"]].head())

    # Try grouping by categorical field, e.g. 'infertility'
    group_field_id = None
    for field in dataset.record_sets[0].fields:
        if field.data_type.lower() in ['boolean','string','text'] and 'infertility' in field.name.lower():
            group_field_id = field.id
            break
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for filtering.")

## 5. Visualization
Visualize the distribution of the selected numeric field, and relationships to categorical fields.

In [ ]:
# Simple visualization using matplotlib (or seaborn)
import matplotlib.pyplot as plt
import seaborn as sns

if age_field_id and age_field_id in df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df[age_field_id], kde=True, bins=8, color='skyblue')
    plt.title(f"Distribution of {age_field_id} in {clinical_rs_id}")
    plt.xlabel("Age at Diagnosis")
    plt.ylabel("Count")
    plt.show()

    # Relationship with infertility, if present
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(6,4))
        sns.boxplot(x=df[group_field_id], y=df[age_field_id])
        plt.title(f"{age_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and process a Croissant-structured dataset using `mlcroissant`. We examined record sets and fields by their `@id`, performed typical EDA on numeric and categorical attributes, and visualized basic relationships. 

The FAIR^2 dataset provides rich tabular and clinical data, but its small, single-site patient cohort calls for caution in generalizing results. Use the dataset for illustrative, academic, or methodological work rather than predictive modeling.